# Build FAISS Index — SigLIP Combined (imagen + texto) — Google Colab

**Qué hace este notebook:**
1. Monta Google Drive y descomprime los pósters
2. Carga metadata filtrando solo películas con poster **y** overview
3. Encodea los pósters con SigLIP (imagen)
4. Encodea los overviews con SigLIP (texto)
5. Promedia ambos vectores y renormaliza
6. Construye el índice FAISS y lo guarda en Drive

**Antes de ejecutar:**
- Activar GPU: `Runtime → Change runtime type → T4 GPU`
- Tener `posters.zip` en `Mi Unidad/Recomendador_Pelis/`

In [ ]:
# Celda 1 — Instalar dependencias
!pip install -q transformers torch torchvision faiss-cpu pandas Pillow huggingface_hub tqdm sentencepiece protobuf

In [ ]:
# Celda 2 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Celda 3 — Descomprimir pósters
import zipfile
from pathlib import Path

ZIP_PATH    = "/content/drive/MyDrive/Recomendador_Pelis/posters/posters.zip"
EXTRACT_DIR = "/content/posters"
POSTERS_DIR = "/content/posters/posters"

Path(EXTRACT_DIR).mkdir(exist_ok=True)

print("Descomprimiendo pósters...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

posters = list(Path(POSTERS_DIR).glob("*.jpg"))
print(f"Pósters disponibles: {len(posters):,}")

In [ ]:
# Celda 4 — Cargar metadata y filtrar
import pandas as pd

METADATA_PATH  = "/content/posters/processed/metadata.csv"
CLIP_META_PATH = "/content/drive/MyDrive/Recomendador_Pelis/index_metadata.csv"

df = pd.read_csv(METADATA_PATH)

# Remap de rutas
df["poster_path"] = df["tmdbId"].apply(
    lambda tid: f"/content/posters/posters/{int(tid)}.jpg"
)

# Filtrar: poster existente + overview no vacío
df = df[
    df["poster_path"].apply(lambda p: Path(p).exists()) &
    df["overview"].notna() &
    (df["overview"].str.strip() != "")
].reset_index(drop=True)

# Agregar tmdb_poster_url desde index_metadata.csv de CLIP (ya debe estar en Drive)
clip_meta = pd.read_csv(CLIP_META_PATH)[["tmdbId", "tmdb_poster_url"]]
df = df.merge(clip_meta, on="tmdbId", how="left")

print(f"Películas con póster y overview: {len(df):,}")
df.head()

In [ ]:
# Celda 5 — Definir SiglipEncoder y MovieIndex
import torch
import numpy as np
import faiss
from PIL import Image
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm

print(f"GPU disponible: {torch.cuda.is_available()}")

MODEL_ID = "google/siglip-base-patch16-224"


class SiglipEncoder:
    def __init__(self, model_id=MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Cargando SigLIP en {self.device}...")
        self.model = AutoModel.from_pretrained(model_id).to(self.device)
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model.eval()

    def _image_features(self, pixel_values):
        out = self.model.get_image_features(pixel_values=pixel_values)
        feats = out.pooler_output if hasattr(out, "pooler_output") else out
        return feats / feats.norm(dim=-1, keepdim=True)

    def _text_features(self, inputs):
        out = self.model.get_text_features(**inputs)
        feats = out.pooler_output if hasattr(out, "pooler_output") else out
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def encode_images_batch(self, image_paths, batch_size=32):
        all_features = []
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Codificando pósters"):
            batch  = image_paths[i : i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch]
            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            all_features.append(self._image_features(inputs["pixel_values"]).cpu().numpy())
        return np.vstack(all_features)

    @torch.no_grad()
    def encode_texts_batch(self, texts, batch_size=32):
        all_features = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Codificando overviews"):
            batch  = texts[i : i + batch_size]
            inputs = self.processor(
                text=batch, return_tensors="pt", padding=True,
                truncation=True, max_length=64
            ).to(self.device)
            all_features.append(self._text_features(inputs).cpu().numpy())
        return np.vstack(all_features)


class MovieIndex:
    def __init__(self, dim=768):
        self.index    = faiss.IndexFlatIP(dim)
        self.metadata = None

    def build(self, embeddings, metadata):
        self.index.add(embeddings.astype(np.float32))
        self.metadata = metadata.reset_index(drop=True)

    def save(self, index_path, metadata_path):
        faiss.write_index(self.index, index_path)
        self.metadata.to_csv(metadata_path, index=False)
        print(f"Índice guardado:   {index_path}")
        print(f"Metadata guardada: {metadata_path}")

In [ ]:
# Celda 6 — Encodear imágenes
encoder = SiglipEncoder()

image_embeddings = encoder.encode_images_batch(df["poster_path"].tolist())
print(f"Shape imagen: {image_embeddings.shape}")

In [ ]:
# Celda 7 — Encodear textos (overviews)
text_embeddings = encoder.encode_texts_batch(df["overview"].tolist())
print(f"Shape texto: {text_embeddings.shape}")

In [ ]:
# Celda 8 — Combinar: promedio + renormalizar
combined = (image_embeddings + text_embeddings) / 2
norms    = np.linalg.norm(combined, axis=1, keepdims=True)
combined = combined / norms

print(f"Shape combined: {combined.shape}")
print(f"Norm media: {np.linalg.norm(combined, axis=1).mean():.4f}  (esperado: 1.0)")

In [ ]:
# Celda 9 — Construir y guardar el índice
INDEX_PATH = "/content/drive/MyDrive/Recomendador_Pelis/faiss_siglip_combined.index"
META_PATH  = "/content/drive/MyDrive/Recomendador_Pelis/index_metadata_siglip_combined.csv"

idx = MovieIndex(dim=combined.shape[1])
idx.build(combined, df)
idx.save(INDEX_PATH, META_PATH)

print(f"\nListo. {len(df):,} películas indexadas (imagen + texto).")

In [ ]:
# Celda 10 — Verificación rápida
sample = df.sample(1, random_state=42).iloc[0]

img   = Image.open(sample["poster_path"]).convert("RGB")
inp_i = encoder.processor(images=img, return_tensors="pt").to(encoder.device)
inp_t = encoder.processor(
    text=[sample["overview"]], return_tensors="pt",
    truncation=True, max_length=64
).to(encoder.device)

with torch.no_grad():
    vec_i = encoder._image_features(inp_i["pixel_values"]).cpu().numpy()
    vec_t = encoder._text_features(inp_t).cpu().numpy()

vec = (vec_i + vec_t) / 2
vec = vec / np.linalg.norm(vec)

scores, indices = idx.index.search(vec.reshape(1, -1).astype(np.float32), 5)
resultados = df.iloc[indices[0]]

print(f"Query: {sample['title']} ({sample['genres']})")
print()
for i, (_, row) in enumerate(resultados.iterrows()):
    print(f"  {i+1}. {row['title']} ({row['genres']})  — score: {scores[0][i]:.4f}")

In [ ]:
# Celda 11 — Descarga directa al navegador (backup)
from google.colab import files
files.download(INDEX_PATH)
files.download(META_PATH)